<a href="https://colab.research.google.com/github/Tejaswimadastu/Generative_AI/blob/main/ChromaDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install chromadb sentence-transformers grok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.3/121.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.7/66.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/

In [2]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.8 MB/s eta 0:00:00


In [3]:
import os
import chromadb
from groq import Groq
from google.colab import userdata


## Setup

In [4]:
chroma_client=chromadb.PersistentClient(path="chroma_db")
collection=chroma_client.get_or_create_collection(name="ABC_Handbook")
groq_client=Groq(api_key=userdata.get('grok_db'))

In [5]:
# --- Document: ABC Employee Handbook (chunked by section) ---
handbook_chunks = [
    "### Company Structure\nABC was founded in 2024 by Mark and John. The company operates in the AI education space with teams across engineering, content, and operations.",
    "### Work Hours\nStandard work hours are 9 AM to 6 PM, Monday through Friday. Flexible start times between 8–10 AM are permitted with manager approval.",
    "### Leave Policy\nEmployees are entitled to 12 casual leaves and 6 sick leaves per year. Leaves do not carry forward to the next year.",
    "### Remote Work\nEmployees must be present in the office on Tuesdays and Wednesdays. Remote work is permitted on all other working days.",
    "### Onboarding\nNew employees are expected to complete the onboarding module within two weeks of joining. The module covers company tools, processes, and code of conduct.",
    "### Learning and Development\nABC provides access to learning platforms. Priority courses are: Python for Data Science, SQL Fundamentals, and Machine Learning Foundations.",
]


In [7]:
collection.upsert(
    documents=handbook_chunks,
    ids=[f'chunk_{i}' for i in range(len(handbook_chunks))]

)
print(f'Indexed {len(handbook_chunks)} chunks')

Indexed 6 chunks


## RAG Query Function

## Build the RAG Prompt

In [13]:
def ask(question: str, top_k: int = 3) -> str:
    # Retrieve relevant chunks
    results = collection.query(
        query_texts=[question],
        n_results=top_k
    )

    retrieved = results["documents"][0]

    # Combine retrieved documents into context
    context = "\n\n".join(retrieved)

    # Build prompt
    prompt = f"""Answer the user question using only the context provided below.
If the answer cannot be found in the context, reply exactly: "I am not sure."
Do not use outside knowledge.

Context:
{context}

Question: {question}
"""

    # Query LLM
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [14]:
print(type(collection))

<class 'chromadb.api.models.Collection.Collection'>


In [18]:
questions=[
    "How many casual leaves and sick leaves do employees get?",
    "Who is founder of ABC Company?",
    "What is the company's revenue last quarter?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print()

Q: How many casual leaves and sick leaves do employees get?
A: Employees are entitled to 12 casual leaves and 6 sick leaves per year.

Q: Who is founder of ABC Company?
A: The founders of ABC Company are Mark and John.

Q: What is the company's revenue last quarter?
A: I am not sure.

